# Delhi UHI pipeline on Colab

This notebook runs every Python step of the project on the Earth Engine exports:

1. LST from the Landsat brightness temperature (E4 checks it against Landsat L2 ST).
2. Segmentation: weak-label tiles, then 4 configurations × 3 seeds (E1/E2), ONNX export and full-city prediction.
3. Hotspot candidates, cooling-model calibration and the map overlay.
4. Cooling-model cross-validation (E6).

**Before you start**

- Run the Earth Engine exports on your laptop (see the README), and wait until `python -m uhi.gee status` shows them all `COMPLETED`. They land in `MyDrive/uhi-geoai/`.
- Put the code where Colab can see it: either push the repo to GitHub and set `REPO_URL`, or upload the `pipeline/` folder to `MyDrive/uhi-geoai-code/pipeline/`.
- Runtime → Change runtime type → **T4 GPU** (or better).

Everything you need to keep is copied back to `MyDrive/uhi-geoai-out/`. Training is resumable: a run whose `metrics.json` is already in Drive is skipped, so if Colab disconnects, run the cells again.

In [ ]:
REPO_URL = ""  # e.g. "https://github.com/<you>/uhi-geoai.git"; leave empty to use the Drive copy
YEARS = [2023, 2024, 2025]
TRAIN_YEARS = [2023, 2024]  # 2025 is held out for the temporal test (E2)
SEEDS = [0, 1, 2]
EPOCHS = 40
CONFIGS = [  # (arch, encoder, loss): the proposed model, its CE ablation, and the two baselines
    ("unetpp", "efficientnet-b4", "focal_dice"),
    ("unetpp", "efficientnet-b4", "ce"),
    ("unet", "resnet34", "ce"),
    ("deeplabv3p", "resnet50", "ce"),
]
CALIBRATION_YEAR = 2024  # the year the web app serves

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess
DRIVE = "/content/drive/MyDrive"
EXPORTS = f"{DRIVE}/uhi-geoai"        # Earth Engine exports (input)
OUT = f"{DRIVE}/uhi-geoai-out"        # everything this notebook keeps
os.makedirs(OUT, exist_ok=True)

if REPO_URL:
    if not os.path.isdir("/content/uhi-geoai"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/uhi-geoai"], check=True)
    PIPE = "/content/uhi-geoai/pipeline"
else:
    PIPE = "/content/pipeline"
    subprocess.run(["rsync", "-a", "--delete", f"{DRIVE}/uhi-geoai-code/pipeline/", PIPE + "/"], check=True)
print("code at", PIPE)

In [ ]:
%cd {PIPE}
!pip install -q -e ".[train]"
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to GPU")

## 1. Copy the exports to local disk

Reading GeoTIFFs straight from Drive is slow, so the exports are copied to `data/` first. Large rasters may arrive as several shards (`s2_2024-0000000000-0000000000.tif`, ...); the pipeline merges them on read.

In [ ]:
!mkdir -p data results
!rsync -a --info=stats1 "{EXPORTS}/" data/
import glob
expected = ["labels_weak", "worldcover30"] + [f"{p}_{y}" for y in YEARS for p in ("s2", "ndvi30", "bt", "st")]
missing = [n for n in expected if not glob.glob(f"data/{n}*.tif")]
print("missing:", missing or "nothing")
assert not missing, "Wait for the Earth Engine exports to finish (python -m uhi.gee status)."

## 2. Land surface temperature

`lst_{year}.tif` from brightness temperature and Sentinel-2 NDVI. E4 (the comparison against Landsat L2 ST) needs the class map, so it runs after segmentation.

In [ ]:
for y in YEARS:
    !python -m uhi.process lst --year {y}

## 3. Segmentation

Tiles are 256 × 256 px at 10 m, split 70/15/15 by 2.56 km block so no test block touches training. Test scores use hand-corrected labels from `tiles/test_corrected/` where they exist; until then `metrics.json` says `partly weak labels`, and those numbers must not be reported as accuracy.

In [ ]:
if os.path.isdir(f"{OUT}/tiles"):
    !rsync -a "{OUT}/tiles/" tiles/
else:
    !python -m uhi.seg.tiles --years {" ".join(map(str, YEARS))} --out tiles
    !rsync -a tiles/ "{OUT}/tiles/"
!ls tiles; cat tiles/stats.json

If you have hand-corrected test tiles (`<row>_<col>.npz` with a `y` array), put them in `MyDrive/uhi-geoai-out/tiles/test_corrected/` and rerun the cell above before training.

In [ ]:
import json, time
os.makedirs(f"{OUT}/runs", exist_ok=True)
for arch, enc, loss in CONFIGS:
    for seed in SEEDS:
        name = f"{arch}_{enc}_{loss}_s{seed}"
        if os.path.exists(f"{OUT}/runs/{name}/metrics.json"):
            print("skip", name, "(already in Drive)")
            continue
        t0 = time.time()
        !python -m uhi.seg.train --arch {arch} --encoder {enc} --loss {loss} --seed {seed} --epochs {EPOCHS} \
            --train-years {" ".join(map(str, TRAIN_YEARS))} --test-years {" ".join(map(str, YEARS))} --out runs
        !rsync -a "runs/{name}/" "{OUT}/runs/{name}/"
        print(f"{name} done in {(time.time() - t0) / 60:.1f} min")

### Results table (mean ± SD over seeds)

Read straight from the `metrics.json` files. These are the only segmentation numbers that go into the paper.

In [ ]:
import numpy as np
rows = []
for arch, enc, loss in CONFIGS:
    ms = [json.load(open(p)) for p in sorted(glob.glob(f"{OUT}/runs/{arch}_{enc}_{loss}_s*/metrics.json"))]
    if not ms:
        continue
    for year in map(str, YEARS):
        tests = [m["test"][year] for m in ms if year in m["test"]]
        if not tests:
            continue
        miou = np.array([t["miou"] for t in tests])
        rows.append((arch, enc, loss, year, len(tests), f"{miou.mean():.3f} ± {miou.std(ddof=1) if len(miou) > 1 else 0:.3f}",
                     tests[0]["labels"]))
for r in rows:
    print(" | ".join(map(str, r)))

## 4. Export the proposed model and classify the city

Picks the proposed configuration's seed with the best **validation** mIoU (never the test score), exports it to ONNX and predicts `classes_{year}.tif` for every year.

In [ ]:
best = max(glob.glob(f"{OUT}/runs/unetpp_efficientnet-b4_focal_dice_s*/metrics.json"),
           key=lambda p: json.load(open(p))["best_val_miou"])
best_run = os.path.dirname(best)
print("using", best_run)
!python -m uhi.seg.infer export --run "{best_run}"
for y in YEARS:
    !python -m uhi.seg.infer predict --onnx "{best_run}/model.onnx" --stats tiles/stats.json --year {y}

## 5. LST check (E4), water-emissivity ablation, hotspots, cooling model

In [ ]:
for y in YEARS:
    !python -m uhi.process lst --year {y} --classes classes_{y}
    !python -m uhi.evaluate lst --year {y}
    for thr in (1.5, 2.0, 2.5):  # thresholds for the E5 stability check
        !python -m uhi.process hotspots --year {y} --min-intensity {thr}
    !python -m uhi.process calibrate --year {y}
    !python -m uhi.evaluate cooling --year {y}
!python -m uhi.process overlay --year {CALIBRATION_YEAR} --out results/layers
!ls results

In [ ]:
for y in YEARS:
    for name in (f"e4_lst_{y}.json", f"e6_cooling_{y}.json", f"cooling_params_{y}.json"):
        p = f"results/{name}"
        if os.path.exists(p):
            print("==", name)
            print(open(p).read()[:1500])

## 6. Save to Drive

`results/` holds the experiment logs, SQL and cooling parameters. The two rasters the database needs (`ndvi30_study_*`, `builtfrac30_*`) and the class maps go to `data/`. Download `uhi-geoai-out/results` and `uhi-geoai-out/data` into `pipeline/results` and `pipeline/data` on your laptop, then run `scripts/load_db.sh 2024`.

In [ ]:
!rsync -a results/ "{OUT}/results/"
!mkdir -p "{OUT}/data"
!rsync -a --include="classes_*.tif" --include="lst_*.tif" --include="suhi_*.tif" --include="ndvi30_study_*.tif" \
    --include="builtfrac30_*.tif" --exclude="*" data/ "{OUT}/data/"
!du -sh "{OUT}"/*